# 03/04 — Responder spot map

For each spot, compute z-shift of a pathology signature relative to that region's WT distribution. Spots with strongly negative z (closer to WT than expected) under BRICHOS = *responder spots*. Map to tissue, ask whether responders cluster regionally or follow a meningeal-to-parenchymal axis.

Output: an updated AnnData with a `PIG_zshift` obs column and a `responder` boolean.

In [ ]:
from __future__ import annotations
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('could not locate project root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# regional-annotation h5ad lives on the processing volume
BASEDIR = Path('/Volumes/processing2/ST_BRICHOS/data')
H5AD_ORIENTED = BASEDIR / 'ST_BRICHOS_region_subcluster_oriented.h5ad'
H5AD_BASE     = BASEDIR / 'ST_BRICHOS_region_subcluster.h5ad'
H5AD = H5AD_ORIENTED if H5AD_ORIENTED.exists() else H5AD_BASE
COUNT_LAYER = 'counts'   # raw integer counts live here, not in .X

TBL = ROOT / 'results' / 'tables' / 'attenuation'
TBL.mkdir(parents=True, exist_ok=True)
FIG = ROOT / 'results' / 'figures' / 'manuscript'
FIG.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY    = 'sample_id'           # change to 'library_id' if obs uses that
REGION_KEY    = 'anatomical_region'   # adjust to your obs column for regions
TREATMENT_KEY = 'treatment'
print('h5ad        :', H5AD)
print('count layer :', COUNT_LAYER)


In [ ]:
from utils.attenuation import responder_zshift

adata = sc.read_h5ad(H5AD)
if 'PIG_score' not in adata.obs.columns:
    PIG_GENES = ['Cst7', 'Itgax', 'Apoe', 'Trem2', 'Tyrobp', 'Lpl',
                 'Csf1', 'Spp1', 'Clec7a', 'Ccl3', 'Ctsb', 'Ctsd']
    present = [g for g in PIG_GENES if g in adata.var_names]
    sc.tl.score_genes(adata, present, score_name='PIG_score',
                      use_raw=False)


### z-shift relative to regional WT

In [ ]:
z = responder_zshift(adata,
                     score_key='PIG_score',
                     treatment_key=TREATMENT_KEY,
                     region_key=REGION_KEY,
                     sample_key=SAMPLE_KEY,
                     ref='WT')
adata.obs['PIG_zshift'] = z
adata.obs['responder']  = adata.obs['PIG_zshift'] < -1.0
adata.obs.groupby([TREATMENT_KEY, REGION_KEY])['responder']\
         .mean().unstack(fill_value=0).round(3)


### Spatial plots — per-treatment composite

In [ ]:
import matplotlib.pyplot as plt
for treat in ['WT', 'PBS', 'BRICHOS']:
    sub = adata[adata.obs[TREATMENT_KEY] == treat]
    if sub.n_obs == 0:
        continue
    libs = sub.obs[SAMPLE_KEY].unique()
    fig, axes = plt.subplots(1, len(libs),
                             figsize=(3 * len(libs), 3.2),
                             squeeze=False)
    for ax, lib in zip(axes.flat, libs):
        sub_l = sub[sub.obs[SAMPLE_KEY] == lib]
        sp = adata.uns['spatial'][lib]
        sf = sp['scalefactors']['tissue_hires_scalef']
        ax.imshow(sp['images']['hires'], origin='upper')
        xy = sub_l.obsm['spatial'] * sf
        c = sub_l.obs['PIG_zshift'].clip(-3, 3).values
        ax.scatter(xy[:, 0], xy[:, 1], c=c, s=0.6,
                   cmap='coolwarm', vmin=-3, vmax=3, edgecolors='none')
        ax.set_title(f'{lib} [{treat}]', fontsize=8, loc='left')
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
    fig.suptitle(f'{treat}: PIG z-shift relative to regional WT',
                 fontsize=10)
    fig.tight_layout()
    fig.savefig(FIG / f'responders_{treat}.svg', bbox_inches='tight')
    fig.savefig(FIG / f'responders_{treat}.png',
                bbox_inches='tight', dpi=200)
    plt.show()


### Quantify: responder fraction per region

Compare BRICHOS vs PBS responder fraction with a per-mouse Mann-Whitney for robustness.

In [ ]:
from scipy.stats import mannwhitneyu
rows = []
for region in adata.obs[REGION_KEY].dropna().unique():
    by_mouse = (adata.obs[adata.obs[REGION_KEY] == region]
                .groupby(SAMPLE_KEY)
                .agg(treatment=(TREATMENT_KEY, 'first'),
                     responder_frac=('responder', 'mean')))
    pbs = by_mouse.loc[by_mouse['treatment'] == 'PBS',
                       'responder_frac'].values
    bri = by_mouse.loc[by_mouse['treatment'] == 'BRICHOS',
                       'responder_frac'].values
    if len(pbs) >= 2 and len(bri) >= 2:
        u, p = mannwhitneyu(bri, pbs, alternative='greater')
    else:
        u, p = (np.nan, np.nan)
    rows.append(dict(region=region, mean_pbs=np.mean(pbs) if len(pbs) else np.nan,
                     mean_bri=np.mean(bri) if len(bri) else np.nan,
                     mwu_p=p))
resp_df = pd.DataFrame(rows).set_index('region').sort_values('mwu_p')
resp_df.to_csv(TBL / 'responder_fraction_by_region.tsv', sep='\t')
resp_df
